In [ ]:
from pathlib import Path

DOCS_DIR = Path("./all_flat")
EXTENSIONS = {".pdf", ".txt", ".docx"}

files = [f for f in DOCS_DIR.iterdir() if f.is_file() and f.suffix.lower() in EXTENSIONS]

print(f"Found {len(files)} documents in {DOCS_DIR}")
for f in sorted(files):
    print(f.name)

In [ ]:
import requests

API_URL = "http://localhost:8070/document/batch/ingest"
BATCH_SIZE = 10

def upload_batch(batch: list[Path]) -> dict:
    file_handles = []
    try:
        file_handles = [open(f, "rb") for f in batch]
        files = [("files", (f.name, fh, "application/octet-stream"))
                 for f, fh in zip(batch, file_handles)]
        resp = requests.post(API_URL, files=files, timeout=120)
        resp.raise_for_status()
        return resp.json()
    finally:
        for fh in file_handles:
            fh.close()

all_results = []
all_errors  = []

batches = [files[i:i+BATCH_SIZE] for i in range(0, len(files), BATCH_SIZE)]
print(f"Sending {len(files)} files in {len(batches)} batches")

for i, batch in enumerate(batches, 1):
    try:
        result = upload_batch(batch)
        all_results.extend(result.get("documents", []))
        all_errors.extend(result.get("errors", []))
        print(f"Batch {i}/{len(batches)}: ok")
    except requests.HTTPError as e:
        print(f"Batch {i}/{len(batches)}: HTTP {e.response.status_code}")
    except Exception as e:
        print(f"Batch {i}/{len(batches)}: error: {e}")

print(f"Sent {len(files)}, ok {len(all_results)}, errors {len(all_errors)}")
for err in all_errors:
    print(err["index"], err["error"])